# 2-GPU DDP throughput TEST (v2 — robust, via `accelerate launch`)

The DDP runs as a SUBPROCESS (`accelerate launch ddp_probe.py`), so none of the
`notebook_launcher` rules apply — the notebook's main process can import anything.
Dummy data, ~10 min, fits a 4h account.

Setup: **Accelerator = GPU T4 x2**, **Internet On**, Secret **HF_TOKEN**.
Run cells top to bottom. Read the `world=... ex/s` lines from cells 5 (2-GPU) and 6 (1-GPU).

In [ ]:
# Cell 1 — deps
import subprocess, sys
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
pip("-U", "transformers>=5.8", "accelerate>=1.2", "peft>=0.14", "bitsandbytes>=0.45", "sentencepiece", "protobuf")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
print("deps installed")

In [ ]:
# Cell 2 — GPU check
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

In [ ]:
# Cell 3 — download base model
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download
login(UserSecretsClient().get_secret("HF_TOKEN"))
snapshot_download(repo_id="google/gemma-4-E2B-it-qat-q4_0-unquantized",
                  local_dir="/kaggle/working/e2b_test",
                  allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "tokenizer*"])
print("model downloaded")

In [ ]:
# Cell 4 — clone repo to get ddp_probe.py
import subprocess
subprocess.run(["git", "clone", "--depth", "1", "--branch", "feat/chess-coach-sft",
                "https://github.com/RyanDev1st/llm-and-engine.git",
                "/kaggle/working/llm-and-engine"],
               env={"GIT_LFS_SKIP_SMUDGE": "1"}, check=False)
print("repo cloned")

In [ ]:
# Cell 5 — 2-GPU data-parallel (the test)
!CHESS_BASE=/kaggle/working/e2b_test accelerate launch --num_processes 2 --multi_gpu /kaggle/working/llm-and-engine/src/llm/llm_training/ddp_probe.py

In [ ]:
# Cell 6 — 1-GPU baseline (for comparison)
!CHESS_BASE=/kaggle/working/e2b_test accelerate launch --num_processes 1 /kaggle/working/llm-and-engine/src/llm/llm_training/ddp_probe.py